# Функциональное программирование

## Домашнее задание 4

Напомним, что типы `А` и `В` *изоморфны*, если существуют функции `f :: A -> B` и `g :: B -> A`, такие что верны равенства `f . g = id` и `g . f = id`.

**1.** Докажите, что для любых типов `А, В` изоморфны типы:
- a) `Ordering` и `Either () Bool`;
- b) `А` и `(А, ())`;
- c) `(Maybe A, Maybe B)` и `Maybe (Either (A, B) (Either A B)).

In [138]:
-- a) Предъявим нужные f и g:
f1a :: Ordering -> Either () Bool
f1a LT = Left ()
f1a EQ = Right False
f1a GT = Right True

g1a :: Either () Bool -> Ordering
g1a (Left ()) = LT
g1a (Right False) = EQ
g1a (Right True) = GT

-- Проверка
:t f1a . g1a
map (f1a . g1a) [Left (), Right False, Right True] == [Left (), Right False, Right True]

:t g1a . f1a
map (g1a . f1a) [LT, EQ, GT] == [LT, EQ, GT]

f1a . g1a :: Either () Bool -> Either () Bool

True

g1a . f1a :: Ordering -> Ordering

True

In [139]:
-- b) Предъявим нужные f и g:
f1b :: a -> (a, ())
f1b x = (x, ())

g1b :: (a, ()) -> a
g1b (x, ()) = x

-- Проверка
:t f1b . g1b
(f1b . g1b) ("hi", ()) == ("hi", ())

:t g1b . f1b
(g1b . f1b) 42 == 42

f1b . g1b :: forall {b}. (b, ()) -> (b, ())

True

g1b . f1b :: forall {c}. c -> c

True

In [140]:
-- c) Предъявим нужные f и g:
f1c :: (Maybe a, Maybe b) -> Maybe (Either (a, b) (Either a b))
f1c (Nothing, Nothing) = Nothing
f1c (Just x, Nothing) = Just (Right (Left x))
f1c (Nothing, Just y) = Just (Right (Right y))
f1c (Just x, Just y) = Just (Left (x, y))

g1c :: Maybe (Either (a, b) (Either a b)) -> (Maybe a, Maybe b)
g1c Nothing = (Nothing, Nothing)
g1c (Just (Left (x, y))) = (Just x, Just y)
g1c (Just (Right (Left x))) = (Just x, Nothing)
g1c (Just (Right (Right y))) = (Nothing, Just y)

-- Проверка
:t f1c . g1c
let qs = [Nothing, Just (Left (1, 'a')), Just (Right (Left 1)), Just (Right (Right 'a'))]
map (f1c . g1c) qs == qs

:t g1c . f1c
let ps = [(Nothing, Nothing), (Just 1, Nothing), (Nothing, Just 'a'), (Just 1, Just 'a')]
map (g1c . f1c) ps == ps

f1c . g1c :: forall {a} {b}. Maybe (Either (a, b) (Either a b)) -> Maybe (Either (a, b) (Either a b))

True

g1c . f1c :: forall {a} {b}. (Maybe a, Maybe b) -> (Maybe a, Maybe b)

True

**2.** Определение натуральных чисел `data Nat = Z | S Nat` весьма неэффективно, поскольку представление числа $n$ имеет размер $O(n)$. Предположим, определен тип:
```
data NatB = ZB | Db NatB | DbI NatB
```

Проинтерпретируйте значения этого типа как натуральные числа таким образом, чтобы запись числа $n$ была размера $O(\log n)$ (тут достаточно неформального объяснения). Реализуйте функции `natb2nat :: NatB -> Nat` и `nat2natb :: Nat -> NatB` в соответствии с вашей интерпретацией и таким образом, что `natb2nat . nat2natb = id`. Получился ли у вас изоморфизм типов `Nat` и `NatB`?

Интерпретация: `ZB` $= 0$, `Db x` $= 2x$, `DbI x` $= 2x + 1$. Это двоичная запись от младших разрядов к старшим, поэтому длина представления числа $n$ есть $O(\log n)$.

Изоморфизма не получилось: представление не единственно — `Db ZB` тоже обозначает `0`, `Db (Db ZB)` — тоже `0` и т. д., то есть разные термы `NatB` отвечают одному `Nat`, и обратная композиция `nat2natb . natb2nat` не является тождественной.

In [141]:
data Nat = Z | S Nat deriving Show
data NatB = ZB | Db NatB | DbI NatB deriving Show

-- Сложение и удвоение в Nat
addN :: Nat -> Nat -> Nat
addN Z m = m
addN (S n) m = S (addN n m)

dblN :: Nat -> Nat
dblN n = addN n n

natb2nat :: NatB -> Nat
natb2nat ZB = Z
natb2nat (Db x) = dblN (natb2nat x)
natb2nat (DbI x) = S (dblN (natb2nat x))

-- Возвращает частное и остаток от деления на 2
halfMod :: Nat -> (Nat, Bool)
halfMod Z = (Z, False)
halfMod (S Z) = (Z, True)
halfMod (S (S n)) = let (q, r) = halfMod n in (S q, r)

nat2natb :: Nat -> NatB
nat2natb Z = ZB
nat2natb n = case halfMod n of
 (q, False) -> Db (nat2natb q)
 (q, True) -> DbI (nat2natb q)

-- Проверка
let five = S (S (S (S (S Z))))
nat2natb five
natb2nat (nat2natb five)

DbI (Db (DbI ZB))

S (S (S (S (S Z))))

**3.** Назовем *отрезком* списка его кусок от $i$-го до $(i + k)$-го элемента включительно. Реализуйте функцию `segs :: [a] -> [[a]]`, выводящую все отрезки данного списка в любом порядке. Например, список `segs "hello"` должен с точностью до перестановки совпадать со списком `["h", "e", "l", "l", "о", "he", "el", "ll", "lo", "hel", "ell", "llo", "hell", "ello", "hello"]`. Если все элементы входного списка были различны, повторов в полученном списке быть не должно. Если входной список бесконечен, допускается любое поведение функции.

In [142]:
-- Все непустые префиксы
prefs :: [a] -> [[a]]
prefs [] = []
prefs (x:xs) = [x] : map (x:) (prefs xs)

-- Все непустые суффиксы
suffs :: [a] -> [[a]]
suffs [] = []
suffs l@(x:xs) = l : suffs xs

-- Отрезок <=> префикс некоторого суффикса
segs :: [a] -> [[a]]
segs xs = concatMap prefs (suffs xs)

-- Проверка
segs "hello"
length (segs "hello")

["h","he","hel","hell","hello","e","el","ell","ello","l","ll","llo","l","lo","o"]

15

**4.** Быть может, используя `zip`, реализуйте функцию `nrem :: Int -> [a] -> [a]`, удаляющую каждый $n$-ый (считая с первого), элемент списка, так что `nrem 3 [1,2,3,4,5,6,7] == [1,2,4,5,7]`. Если входной список бесконечен, допускается любое поведение функции.

In [143]:
nrem :: Int -> [a] -> [a]
nrem n xs = [x | (x, i) <- zip xs [1..], i `mod` n /= 0]

-- Проверка
nrem 3 [1,2,3,4,5,6,7]
nrem 2 "abcdefgh"
nrem 1 [1,2,3,4,5]

[1,2,4,5,7]

"aceg"

[]

**5.** Реализуйте функцию, устраняющую в списке все повторы (необязательно идущие подряд). Какая из повторяющихся копий оставляется, решите сами. Если входной список бесконечен, допускается любое поведение функции.

In [144]:
-- Оставляем первое вхождение каждого элемента
delDup :: Eq a => [a] -> [a]
delDup [] = []
delDup (x:xs) = x : delDup [y | y <- xs, y /= x]

-- Проверка
delDup [1,2,3,2,1,4,3,5]
delDup "nigga g"

[1,2,3,4,5]

"niga "

**6.** Реализуйте функцию `part :: Int -> Int -> [[Int]]`, т. ч. `part m n` есть список всех разбиений `[x1,...,xm]` числа $n \geq 0$ в сумму $m > 0$ целых неотрицательных слагаемых. Порядок слагаемых в разбиении важен, а порядок разбиений в выводе функции — нет.

In [145]:
part :: Int -> Int -> [[Int]]
part 1 n = [[n]]
part m n = [x : rest | x <- [0..n], rest <- part (m - 1) (n - x)]

-- Проверка
part 1 3
part 3 2

[[3]]

[[0,0,2],[0,1,1],[0,2,0],[1,0,1],[1,1,0],[2,0,0]]

**7.** Используя `foldl` и `foldr`, реализуйте библиотечные функции:
- a) `map`;
- b) `filter`;
- c) `all`; *(постарайтесь не использовать ни одной локальной переменной типа элемента списка, а применять оператор композиции)*
- d) `any`. *(тоже)*

In [146]:
-- a)
map' :: (a -> b) -> [a] -> [b]
map' f = foldr (\x acc -> f x : acc) []

-- Проверка
map' (+1) [1,2,3,4]
map' show [1,2,3]

Line 3: Use map
Found:
foldr (\ x acc -> f x : acc) []
Why not:
map (\ x -> f x)

[2,3,4,5]

["1","2","3"]

In [147]:
-- b)
filter' :: (a -> Bool) -> [a] -> [a]
filter' p = foldr (\x acc -> if p x then x : acc else acc) []

-- Проверка
filter' even [1..10]

[2,4,6,8,10]

In [148]:
-- c)
all' :: (a -> Bool) -> [a] -> Bool
all' p = foldr ((&&) . p) True

-- Проверка
all' even [2,4,6]
all' even [2,4,5]
all' even []

True

False

True

In [149]:
-- d)
any' :: (a -> Bool) -> [a] -> Bool
any' p = foldr ((||) . p) False

-- Проверка
any' even [1,3,5]
any' even [1,3,4]
any' even []

False

True

False

**8.** Реализуйте библиотечные функции (прежде опишите словесно, что они делают):
- a) `takeWhile`;
- b) `dropWhile`.

**a)** `takeWhile p xs` берёт из `xs` самый длинный префикс, все элементы которого удовлетворяют предикату `p`.

In [150]:
takeWhile' :: (a -> Bool) -> [a] -> [a]
takeWhile' _ [] = []
takeWhile' p (x:xs)
 | p x = x : takeWhile' p xs
 | otherwise = []

-- Проверка
takeWhile' (< 4) [1,2,3,4,5,1,2]
takeWhile' even [2,4,6,7,8]

[1,2,3]

[2,4,6]

**b)** `dropWhile p xs` отбрасывает у `xs` максимальный префикс, все элементы которого удовлетворяют `p`, и возвращает оставшийся суффикс.

In [151]:
-- b) dropWhile p xs отбрасывает у xs максимальный префикс,
-- все элементы которого удовлетворяют p, и возвращает оставшийся хвост.
dropWhile' :: (a -> Bool) -> [a] -> [a]
dropWhile' _ [] = []
dropWhile' p l@(x:xs)
  | p x = dropWhile' p xs
  | otherwise = l

-- Проверка
dropWhile' (< 4) [1,2,3,4,5,1,2]
dropWhile' even [2,4,6,7,8]

[4,5,1,2]

[7,8]

**9.** Библиотечная функция `foldr1` отличается от `foldr` тем, что не использует начального значения. Опишите словесно, как она работает, и реализуйте ее с помощью `foldr`. Используя `foldr1`, реализуйте функцию `lmax :: Ord a => [a] -> а`, которая ищет максимальный элемент в непустом списке.

`foldr1 f [x1, x2, ..., xn]` вычисляет `f x1 (f x2 (... (f x_{n-1} xn) ...))` — свёртка вправо, роль начального значения играет последний элемент. На пустом списке не определена.

In [152]:
foldr1' :: (a -> a -> a) -> [a] -> a
foldr1' f = unwrap . foldr step Nothing
 where
  step x Nothing = Just x
  step x (Just y) = Just (f x y)
  unwrap Nothing = error "foldr1': empty list"
  unwrap (Just v) = v

lmax :: Ord a => [a] -> a
lmax = foldr1' max

-- Проверка
foldr1' (+) [1,2,3,4,5]
-- error:
-- foldr1' []
lmax [3, 1, 4, 1, 5, 9, 2, 6]
lmax "hello"

15

9

'o'

**10.** Реализуйте функцию, возвращающую список всех префиксов строки-аргумента. Если входная строка бесконечна, допускается любое поведение функции.

In [153]:
prefixes :: [a] -> [[a]]
prefixes [] = [[]]
prefixes (x:xs) = [] : map (x:) (prefixes xs)

-- Проверка
prefixes "abc"
prefixes [1,2,3,4]

["","a","ab","abc"]

[[],[1],[1,2],[1,2,3],[1,2,3,4]]

**11.** Реализуйте функцию `rotts :: [a] -> [[a]]`, возвращающую всевозможные циклические сдвиги (перестановки) данного списка, с помощью:
- a) `iterate`;
- b) `scanl`.

В частности, должно быть `rotts [1,2,3] = [[1,2,3], [2,3,1], [3,1,2]]` с точностью до перестановки "внешнего" списка. Если входной список бесконечен, допускается любое поведение функции.

In [154]:
-- Один циклический сдвиг влево
rot1 :: [a] -> [a]
rot1 [] = []
rot1 (x:xs) = xs ++ [x]

-- Проверка
rot1 [1,2,3,4]

[2,3,4,1]

In [155]:
-- a)
rottsA :: [a] -> [[a]]
rottsA xs = take (length xs) (iterate rot1 xs)

-- Проверка
rottsA [1,2,3]
rottsA "abcd"

[[1,2,3],[2,3,1],[3,1,2]]

["abcd","bcda","cdab","dabc"]

In [156]:
-- b)
rottsB :: [a] -> [[a]]
rottsB xs = scanl (\acc _ -> rot1 acc) xs (drop 1 xs)

-- Проверка
rottsB [1,2,3]
rottsB "abcd"

[[1,2,3],[2,3,1],[3,1,2]]

["abcd","bcda","cdab","dabc"]

**12.** Реализуйте библиотечную функцию `unzip :: [(a,b)] -> ([a],[b])`, прежде описав словесно, что она делает, используя:
- a) `map`;
- b) `foldr`.

`unzip` по списку пар возвращает пару списков: в первом собраны все первые компоненты пар в том же порядке, во втором — все вторые.

In [157]:
-- a)
unzipA :: [(a, b)] -> ([a], [b])
unzipA ps = (map fst ps, map snd ps)

-- Проверка
unzipA [(1,'a'), (2,'b'), (3,'c')]

([1,2,3],"abc")

In [158]:
-- b)
unzipB :: [(a, b)] -> ([a], [b])
unzipB = foldr (\(x, y) (xs, ys) -> (x : xs, y : ys)) ([], [])

-- Проверка
unzipB [(1,'a'), (2,'b'), (3,'c')]

([1,2,3],"abc")